# 01 · Evaluación de briefings reales (camino 1)

Generamos **N = 6 briefings reales** (3 carteras ficticias + 3 listas de tickers, una con el PDF y el gráfico de
ejemplo) y medimos latencia, coste, *grounding* de cifras, frases con recomendación, fuentes por punto clave,
duración del podcast, caídas a sustituto y un **LLM-juez** (Claude Sonnet, rúbrica 1-5).
Incluye la **revisión de prompts con 3 carteras** (tarea M de `docs/05_roadmap_TODO.md`).

La lógica vive en `scripts/evaluar_briefings.py`; los resultados ligeros en `notebooks/eval/` (JSON/CSV/MD, sin
audio ni claves). Los briefings completos quedan en `data/outputs/<id>/` (no se versionan).
Ejecución real: 06-oct-2026 entre las 13:45 y las 13:51 (sesión europea abierta: precios **intradía**, no de cierre).

**Coste.** Con `RUN_REAL = False` (por defecto) el cuaderno **solo carga** lo cacheado en `notebooks/eval/` y no
gasta nada. Con `RUN_REAL = True` genera/juzga solo lo que falte, con un tope de 1,2 € medido con
`StepMetric.est_cost_eur` (se puede activar sin editar con la variable de entorno `BRIEFLY_EVAL_RUN_REAL=1`).
Configuración barata: edge-tts, sin verificación del podcast, sin FinBERT, CLIP ni portada, sin vídeo ni Telegram.

In [1]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT / "src")]
os.chdir(ROOT)  # rutas relativas de .env (data/outputs, data/cache)

RUN_REAL = os.getenv("BRIEFLY_EVAL_RUN_REAL", "0") == "1"   # False por defecto: solo carga resultados
BUDGET_EUR = 1.2

import evaluar_briefings as ev
ev.configure_env()          # fija la configuración barata ANTES de usar briefer
import pandas as pd
pd.set_option("display.max_colwidth", 80)
print("RUN_REAL =", RUN_REAL, "· tickers fuera del universo:", ev.check_scenarios() or "ninguno")

RUN_REAL = False · tickers fuera del universo: ninguno


Los 6 escenarios: tres carteras con pesos (en memoria, nunca en disco: ADR-005) y tres listas de tickers.

In [2]:
pd.DataFrame([{"escenario": s.name, "tipo": s.kind, "tickers": ", ".join(s.tickers) or "—",
               "cartera (peso)": ", ".join(f"{t} {w:.0%}" for t, w in s.portfolio) if s.portfolio else "—",
               "subidas": ", ".join(Path(u).name for u in s.uploads) or "—"} for s in ev.SCENARIOS])

,escenario,tipo,tickers,cartera (peso),subidas
0,cartera_banca_es,cartera,—,"SAN.MC 40%, BBVA.MC 35%, CABK.MC 25%",—
1,cartera_tech_usa,cartera,—,"AAPL 40%, MSFT 30%, NVDA 30%",—
2,cartera_mixta_defensiva,cartera,—,"IBE.MC 50%, ITX.MC 30%, AAPL 20%",—
3,tickers_ibex_varios,tickers,"TEF.MC, REP.MC, AENA.MC",—,—
4,tickers_usa_megacaps,tickers,"AMZN, GOOGL, META, TSLA",—,—
5,tickers_con_pdf_y_grafico,tickers+subidas,"FER.MC, AMS.MC",—,"resultados_ejemplo.pdf, grafico_ejemplo.png"


## 1 · Generación

`pipeline.run_briefing(..., mode="real", use_cache=True, deliver=[], make_video=False)`. Cada briefing se evalúa
en memoria justo al terminar (con la cartera aún presente) y se añade a `notebooks/eval/briefings.jsonl`;
los escenarios ya presentes no se regeneran.

In [3]:
records = ev.generate_all(RUN_REAL, BUDGET_EUR)
ok = [r for r in records if r.get("id")]
print(f"{len(ok)} briefings válidos · {len(records) - len(ok)} fallidos")
pd.DataFrame([{"escenario": r["scenario"], "id": r.get("id"), "error": r.get("error")} for r in records])

6 briefings válidos · 0 fallidos


,escenario,id,error
0,cartera_banca_es,20261006-134527-44869c,None
1,cartera_tech_usa,20261006-134637-845111,None
2,cartera_mixta_defensiva,20261006-134731-ef9961,None
3,tickers_ibex_varios,20261006-134823-89a7d2,None
4,tickers_usa_megacaps,20261006-134913-9f8aba,None
5,tickers_con_pdf_y_grafico,20261006-135000-2ceb28,None


## 2 · Métricas por briefing

- **Cifras trazables**: `guardrails.untraceable_figures` del análisis contra el contexto (`grounding_reference`) y
  del guion contra el análisis (`scriptwriter.build_user_message`), igual que hacen los agentes.
- **Recomendaciones**: frases donde salta `guardrails.contains_advice` (objetivo 0).
- **Fuentes**: puntos clave con ≥ 1 fuente y fuentes que no existen en `context.news`/`insights`.
- **Podcast**: duración del MP3 frente a la banda 3-5 min. **Reintentos/sustitutos**: de `StepMetric.detail`/`error`.

In [4]:
def fila(r):
    fa, fs = r["figures_analysis"], r["figures_script"]
    return {
        "escenario": r["scenario"], "pared s": r["wall_s"], "suma pasos s": r["steps_sum_s"],
        "coste €": round(r["cost_eur"], 4), "noticias": r["n_news"],
        "cifras análisis (traz.)": f'{fa["n"] - len(fa["untraceable"])}/{fa["n"]}',
        "cifras guion (traz.)": f'{fs["n"] - len(fs["untraceable"])}/{fs["n"]}',
        "recom. análisis": len(r["advice_analysis"]), "recom. guion": len(r["advice_script"]),
        "puntos clave": r["n_key_points"], "con fuente": r["kp_with_source"], "fuentes inválidas": r["kp_sources_invalid"],
        "podcast min": round(r["podcast_s"] / 60, 2) if r["podcast_s"] else None, "en 3-5 min": r["podcast_in_band"],
        "reint. analista": r["analyst_retries"], "reint. guionista": r["scriptwriter_retries"],
        "sustitutos": len(r["fallbacks"]), "errores": len(r["errors"]),
    }
tabla = pd.DataFrame([fila(r) for r in ok])
tabla

,escenario,pared s,suma pasos s,coste €,noticias,cifras análisis (traz.),cifras guion (traz.),recom. análisis,recom. guion,puntos clave,con fuente,fuentes inválidas,podcast min,en 3-5 min,reint. analista,reint. guionista,sustitutos,errores
0,cartera_banca_es,70.75,77.17,0.0339,18,27/27,21/21,0,0,6,6,0,3.55,True,0,0,0,0
1,cartera_tech_usa,53.66,54.48,0.0406,18,20/20,19/19,0,0,6,6,0,3.29,True,0,1,0,0
2,cartera_mixta_defensiva,51.79,51.97,0.0336,20,18/18,17/17,0,0,6,6,0,3.47,True,0,0,0,0
3,tickers_ibex_varios,49.83,52.48,0.0327,17,18/18,17/17,0,0,6,6,0,3.51,True,0,0,0,0
4,tickers_usa_megacaps,47.20,53.29,0.0319,20,19/19,19/19,0,0,5,5,0,3.68,True,0,0,0,0
5,tickers_con_pdf_y_grafico,60.48,81.58,0.0691,17,23/23,14/14,0,0,6,6,0,3.17,True,0,0,0,0


Notas de calidad que dejaron los agentes (`StepMetric.detail`): qué corrigieron los guardarraíles antes del resultado final.

In [5]:
pd.DataFrame([{"escenario": r["scenario"], "analista": r["analyst_detail"], "guionista": r["scriptwriter_detail"][:240]} for r in ok])

,escenario,analista,guionista
0,cartera_banca_es,grounding: todas las cifras trazables,"guion: cifras trazables al análisis · guion: 16 intervenciones, ~3.8 min"
1,cartera_tech_usa,grounding: todas las cifras trazables,guion: reintento 1 (Faltan puntos clave del análisis: «Próximos resultados y...
2,cartera_mixta_defensiva,grounding: todas las cifras trazables,"guion: cifras trazables al análisis · guion: 16 intervenciones, ~3.6 min"
3,tickers_ibex_varios,grounding: todas las cifras trazables,"guion: cifras trazables al análisis · guion: 18 intervenciones, ~3.4 min"
4,tickers_usa_megacaps,grounding: todas las cifras trazables,guion: cifras trazables al análisis · guion: 1 frase(s) con causa no matizad...
5,tickers_con_pdf_y_grafico,grounding: todas las cifras trazables,"guion: cifras trazables al análisis · guion: 16 intervenciones, ~3.3 min"


## 3 · Agregados: p50/p95 de pared y coste

Percentiles con interpolación lineal (`scripts/metrics_report.percentile`, tipo 7). La pared medida aquí es
`time.perf_counter()` alrededor de `run_briefing`; se contrasta con la aproximación de `metrics_report.py`
(desde el id y `created_at`). Con N = 6 el p95 es casi el máximo: orientativo.

In [6]:
import metrics_report as mr
rep = mr.build_report(mr.collect([ev.ROOT / "data" / "outputs" / r["id"] for r in ok]), "real") if ok else None
def st(vals):
    vals = [v for v in vals if v is not None]
    return {"N": len(vals), "p50": mr.percentile(vals, 50), "p95": mr.percentile(vals, 95),
            "media": sum(vals) / len(vals) if vals else None, "mín": min(vals, default=None), "máx": max(vals, default=None)}
agg = pd.DataFrame({
    "pared medida (s)": st([r["wall_s"] for r in ok]),
    "pared aprox. metrics_report (s)": st([b["wall_clock_s"] for b in rep["briefings"]]) if rep else {},
    "suma de pasos (s)": st([r["steps_sum_s"] for r in ok]),
    "coste (€)": st([r["cost_eur"] for r in ok]),
    "coste sin subidas (€)": st([r["cost_eur"] for r in ok if not r["uploads"]]),
}).T.round(4)
agg

,N,p50,p95,media,mín,máx
pared medida (s),6.0,52.7250,68.1825,55.6183,47.2000,70.7500
pared aprox. metrics_report (s),6.0,53.4160,68.3458,56.0132,47.3890,70.8350
suma de pasos (s),6.0,53.8850,80.4775,61.8283,51.9700,81.5800
coste (€),6.0,0.0337,0.0620,0.0403,0.0319,0.0691
coste sin subidas (€),5.0,0.0336,0.0392,0.0345,0.0319,0.0406


Latencia y coste **por paso** (desde `Briefing.metrics`, vía `metrics_report.build_report`).

In [7]:
pasos = pd.DataFrame([{"paso": s["step"], "proveedor": ", ".join(s["providers"]), "N": s["latency_s"]["n"],
                       "lat p50 s": s["latency_s"]["p50"], "lat p95 s": s["latency_s"]["p95"],
                       "coste p50 €": s["cost_eur"]["p50"], "coste p95 €": s["cost_eur"]["p95"],
                       "coste total €": s["cost_eur"]["total"]} for s in rep["steps"]]).round(4) if rep else pd.DataFrame()
pasos

,paso,proveedor,N,lat p50 s,lat p95 s,coste p50 €,coste p95 €,coste total €
0,ingest.news,yfinance+rss,6,5.5813,8.4043,0.0000,0.0000,0.0000
1,ingest.prices,yfinance,6,1.7916,17.4264,0.0000,0.0000,0.0000
2,ingest.tickers,local,6,0.0074,0.0091,0.0000,0.0000,0.0000
3,agents.analyst,anthropic/claude-sonnet-5-5,6,14.2204,14.6190,0.0246,0.0263,0.1489
4,agents.scriptwriter,anthropic/claude-haiku-4-5-20251001,6,11.6400,18.5433,0.0084,0.0146,0.0584
5,media.podcast,edge/edge-tts,6,17.1362,22.1542,0.0000,0.0000,0.0000
6,media.transcript,local,6,0.0034,0.0044,0.0000,0.0000,0.0000
7,media.charts,matplotlib,6,1.5380,2.0275,0.0000,0.0000,0.0000
8,storage.save,local,6,0.0063,0.0071,0.0000,0.0000,0.0000
9,ingest.pdf,anthropic/claude-sonnet-5-5,1,18.3648,18.3648,0.0197,0.0197,0.0197


## 4 · LLM-juez (Claude Sonnet, rúbrica fija 1-5)

Criterios: **fidelidad** a las noticias del contexto, **claridad** del guion, **sin consejo** de inversión y
**utilidad** para un inversor minorista. Entrada: contexto resumido (precios con su evolución mensual, titulares
+ extractos ≤ 200 caracteres, documentos), análisis y guion. Salida estructurada (`response_model` propio) con
una frase por criterio. Temperatura: el proveedor de Anthropic del repo no expone `temperature` (Sonnet 5.5 usa
razonamiento adaptativo); se baja el esfuerzo a `effort="low"` para el juez. Resultados en `notebooks/eval/juez.json`.

*Nota:* una primera ronda (`juez_v1_sin_tendencia.json`, 0,087 €) no le pasaba al juez la evolución mensual de
precios que sí ve el Analista, y penalizaba como «no trazables» cifras correctas («−5,17 % desde el 7/09»). Se
corrigió la entrada y se repitió; su coste cuenta en el gasto total.

In [8]:
jueces = ev.judge_all(RUN_REAL, ok, BUDGET_EUR)
juez = pd.DataFrame([{"escenario": v["scenario"], **{c: v[c]["score"] for c in ev.CRITERIA}, "coste €": v["cost_eur"]}
                     for v in jueces.values()])
display(juez)
if len(juez):
    print("Media por criterio:", juez[list(ev.CRITERIA)].mean().round(2).to_dict(), f"· coste juez {juez['coste €'].sum():.4f} €")

,escenario,fidelidad,claridad,sin_consejo,utilidad,coste €
0,cartera_banca_es,3,4,5,4,0.014560
1,cartera_tech_usa,4,4,3,4,0.013868
2,cartera_mixta_defensiva,3,4,4,3,0.014329
3,tickers_ibex_varios,3,4,4,3,0.013686
4,tickers_usa_megacaps,3,4,4,3,0.014783
5,tickers_con_pdf_y_grafico,4,4,5,3,0.015095


Media por criterio: {'fidelidad': 3.33, 'claridad': 4.0, 'sin_consejo': 4.17, 'utilidad': 3.33} · coste juez 0.0863 €


In [9]:
pd.DataFrame([{"escenario": v["scenario"], "criterio": c, "nota": v[c]["score"], "justificación": v[c]["justification"]}
              for v in jueces.values() for c in ev.CRITERIA])

,escenario,criterio,nota,justificación
0,cartera_banca_es,fidelidad,3,"Casi todo es trazable, pero el guion afirma que ""según los titulares"" el imp..."
1,cartera_banca_es,claridad,4,"Estructura de diálogo natural con transiciones sencillas, un valor por bloqu..."
2,cartera_banca_es,sin_consejo,5,"Solo informa, sin compra/venta ni precios objetivo, y añade un aviso explíci..."
3,cartera_banca_es,utilidad,4,"Cubre los tres valores de la cartera con precio, variación mensual y la noti..."
4,cartera_tech_usa,fidelidad,4,"Casi todo es trazable al contexto y se cuida atribuir y no inventar causas, ..."
5,cartera_tech_usa,claridad,4,Estructura clara por valor con transiciones naturales y explicación de «ampl...
6,cartera_tech_usa,sin_consejo,3,"No hay compra/venta y el precio objetivo se atribuye a BNP, pero «os debería..."
7,cartera_tech_usa,utilidad,4,"Prioriza AAPL, MSFT y NVDA con cifras diarias y de periodo, admite cuándo no..."
8,cartera_mixta_defensiva,fidelidad,3,"Casi todo es trazable, pero hay errores: se dice que Jefferies ""mantiene"" la..."
9,cartera_mixta_defensiva,claridad,4,Estructura ordenada por valor con transiciones naturales y atribuciones clar...


## 5 · Revisión de prompts con 3 carteras (tarea M)

Para cada cartera: 0 frases con recomendación (análisis y guion), 0 cifras no trazables y que **cada posición**
aparece en el análisis (por ticker en `KeyPoint.tickers` o por nombre/alias de `TICKER_UNIVERSE` en el texto).

In [10]:
carteras = []
for r in ok:
    if not r["portfolio_mentions"]:
        continue
    pm = r["portfolio_mentions"]
    carteras.append({
        "cartera": r["scenario"], "recom. (an.+guion)": len(r["advice_analysis"]) + len(r["advice_script"]),
        "cifras no trazables (an.+guion)": len(r["figures_analysis"]["untraceable"]) + len(r["figures_script"]["untraceable"]),
        "posiciones en análisis": f'{sum(v["analisis"] for v in pm.values())}/{len(pm)}',
        "posiciones en guion": f'{sum(v["guion"] for v in pm.values())}/{len(pm)}',
        "ausentes del análisis": ", ".join(t for t, v in pm.items() if not v["analisis"]) or "—",
        "titular": r["headline"],
    })
pd.DataFrame(carteras)

,cartera,recom. (an.+guion),cifras no trazables (an.+guion),posiciones en análisis,posiciones en guion,ausentes del análisis,titular
0,cartera_banca_es,0,0,3/3,3/3,—,"Santander lidera el rebote de la banca española, que impulsa al Ibex 35 un 1..."
1,cartera_tech_usa,0,0,3/3,3/3,—,NVIDIA y Microsoft lideran el avance de la sesión mientras Apple cede ligera...
2,cartera_mixta_defensiva,0,0,3/3,3/3,—,Iberdrola lidera las subidas de tu cartera en un Ibex con impulso de la banca


## 6 · Tabla resumen y ficheros de resultados

In [11]:
from datetime import date
def media_juez(c):
    return round(sum(v[c]["score"] for v in jueces.values()) / len(jueces), 2) if jueces else None
figs_n = sum(r["figures_analysis"]["n"] + r["figures_script"]["n"] for r in ok)
figs_bad = sum(len(r["figures_analysis"]["untraceable"]) + len(r["figures_script"]["untraceable"]) for r in ok)
a = agg.to_dict("index")
resumen = {
    "N briefings": len(ok),
    "pared p50 / p95 (s)": f'{a["pared medida (s)"]["p50"]:.1f} / {a["pared medida (s)"]["p95"]:.1f}',
    "coste p50 / p95 (€)": f'{a["coste (€)"]["p50"]:.4f} / {a["coste (€)"]["p95"]:.4f}',
    "cifras trazables (%)": round(100 * (1 - figs_bad / figs_n), 1) if figs_n else None,
    "frases con recomendación": sum(len(r["advice_analysis"]) + len(r["advice_script"]) for r in ok),
    "pasos caídos a sustituto": sum(len(r["fallbacks"]) for r in ok),
    "pasos con error": sum(len(r["errors"]) for r in ok),
    "podcasts en 3-5 min": f'{sum(r["podcast_in_band"] for r in ok)}/{len(ok)}',
    "puntos clave con fuente": f'{sum(r["kp_with_source"] for r in ok)}/{sum(r["n_key_points"] for r in ok)}',
    "fuentes inválidas": sum(r["kp_sources_invalid"] for r in ok),
    **{f"juez {c} (media 1-5)": media_juez(c) for c in ev.CRITERIA},
    "gasto total evaluación (€)": round(ev.spent_eur(), 4),
}
res_df = pd.DataFrame(resumen.items(), columns=["métrica", "valor"])
tabla.to_csv(ev.EVAL_DIR / "por_briefing.csv", index=False, encoding="utf-8")
pasos.to_csv(ev.EVAL_DIR / "por_paso.csv", index=False, encoding="utf-8")
juez.to_csv(ev.EVAL_DIR / "juez.csv", index=False, encoding="utf-8")
md_txt = [f"# Evaluación de briefings reales ({date.today():%d-%m-%Y})", "",
          "Generado por `notebooks/01_evaluacion_briefings.ipynb`.", "", res_df.to_markdown(index=False), "",
          "## Por briefing", "", tabla.to_markdown(index=False), "", "## Agregados", "", agg.to_markdown(), "",
          "## Por paso", "", pasos.to_markdown(index=False), "", "## Juez", "", juez.to_markdown(index=False), "",
          "## Carteras", "", pd.DataFrame(carteras).to_markdown(index=False), "",
          "Ids: " + ", ".join(f'{r["scenario"]}={r["id"]}' for r in ok)]
(ev.EVAL_DIR / "resumen.md").write_text("\n".join(md_txt), encoding="utf-8")
res_df

,métrica,valor
0,N briefings,6
1,pared p50 / p95 (s),52.7 / 68.2
2,coste p50 / p95 (€),0.0337 / 0.0620
3,cifras trazables (%),100.0
4,frases con recomendación,0
5,pasos caídos a sustituto,0
6,pasos con error,0
7,podcasts en 3-5 min,6/6
8,puntos clave con fuente,35/35
9,fuentes inválidas,0


## Conclusiones en 5 líneas para el pitch

1. **6 de 6 briefings reales sin fallos ni sustitutos**: pared p50 **53 s** / p95 **68 s**; coste p50 **0,034 €** / p95 **0,062 €** (con PDF + gráfico, 0,069 €).
2. **Grounding verificable**: el **100 %** de las 232 cifras (análisis y guion) es trazable al contexto, y **35/35** puntos clave citan una fuente real.
3. **Compliance**: **0** frases con recomendación (`contains_advice`) en las 6 y en las 3 carteras, que además mencionan todas sus posiciones (9/9); el juez da 4,2/5 en «sin consejo».
4. **LLM-juez (Sonnet)**: claridad **4,0**, fidelidad **3,3**, utilidad **3,3**. Los fallos son de matiz: causas o atribuciones más firmes en el guion que en el análisis, algo que las puertas numéricas no ven.
5. Siguiente paso: una puerta de atribución y causalidad en el guion y un saludo que dependa de la hora real. Evaluar con el juez cuesta ≈ 0,014 € por briefing: la calidad se puede medir a diario.

*Revisión de prompts con 3 carteras (tarea M):* sin recomendaciones ni cifras inventadas en las 3, y `contains_advice` no salta. Los problemas de matiz están en `notebooks/eval/juez.json`. Algunos ejemplos: «os debería preocupar»; «la edición de noche con el cierre del día» a las 13:45; «la lanzamiento».
